In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
meas_default_o3 = [
    "saleae_out/widgets_dma2d_gcc_o3.csv",
    "saleae_out/screen_fill_dma2d_gcc_o3.csv",
    "saleae_out/screen_blend_dma2d_gcc_o3.csv",
    "saleae_out/bitmap_dma2d_gcc_o3.csv",
    "saleae_out/bitmap_blend_dma2d_gcc_o3.csv",
    "saleae_out/screen_text_dma2d_gcc_o3.csv",
]

meas_default_os = [
    "saleae_out/widgets_dma2d_gcc_os.csv",
    "saleae_out/screen_fill_dma2d_gcc_os.csv",
    "saleae_out/screen_blend_dma2d_gcc_os.csv",
    "saleae_out/bitmap_dma2d_gcc_os.csv",
    "saleae_out/bitmap_blend_dma2d_gcc_os.csv",
    "saleae_out/screen_text_dma2d_gcc_os.csv",
]

meas_improv_o3 = [
    "saleae_out/widgets_dma2d_improved_gcc_o3.csv",
    "saleae_out/screen_fill_dma2d_improved_gcc_o3.csv",
    "saleae_out/screen_blend_dma2d_improved_gcc_o3.csv",
    "saleae_out/bitmap_dma2d_improved_gcc_o3.csv",
    "saleae_out/bitmap_blend_dma2d_improved_gcc_o3.csv",
    "saleae_out/screen_text_dma2d_improved_gcc_o3.csv",
]

meas_improv_os = [
    "saleae_out/widgets_dma2d_improved_gcc_os.csv",
    "saleae_out/screen_fill_dma2d_improved_gcc_os.csv",
    "saleae_out/screen_blend_dma2d_improved_gcc_os.csv",
    "saleae_out/bitmap_dma2d_improved_gcc_os.csv",
    "saleae_out/bitmap_blend_dma2d_improved_gcc_os.csv",
    "saleae_out/screen_text_dma2d_improved_gcc_os.csv",
]

In [ ]:
def calculate_average_pulse_duration(csv_path, num_pulses_to_average=20):
    """
    Reads a CSV file from a Saleae export, finds up to a specified number of 
    high pulses on 'Channel 1', and calculates their average duration.

    Args:
        csv_path (str): The path to the CSV file.
        num_pulses_to_average (int): The maximum number of pulses to average.

    Returns:
        float: The average duration of the pulses in milliseconds. Returns 0 
               if no complete pulses are found or the file doesn't exist.
    """
    if not os.path.exists(csv_path):
        print(f"Warning: File not found at {csv_path}. Skipping.")
        return 0

    data = pd.read_csv(csv_path)
    data.columns = data.columns.str.strip()

    time_s = data['Time [s]']
    channel_data = data['Channel 1']

    # Find the indices of all rising edges (0 -> 1) and falling edges (1 -> 0)
    rising_indices = channel_data.index[(channel_data == 1) & (channel_data.shift(1) == 0)]
    falling_indices = channel_data.index[(channel_data == 0) & (channel_data.shift(1) == 1)]

    if rising_indices.empty or falling_indices.empty:
        print(f"Warning: No rising or falling edges found in {csv_path}.")
        return 0

    pulse_durations = []
    # Iterate through the first `num_pulses_to_average` rising edges found
    for start_idx in rising_indices[:num_pulses_to_average]:
        # Find the first falling edge that occurs *after* the current rising edge
        valid_falling_indices = falling_indices[falling_indices > start_idx]
        
        if not valid_falling_indices.empty:
            end_idx = valid_falling_indices[0]
            
            # Get the corresponding times and calculate duration in milliseconds
            start_time = time_s[start_idx]
            end_time = time_s[end_idx]
            duration_ms = (end_time - start_time) * 1000
            pulse_durations.append(duration_ms)
        else:
            # No more corresponding falling edges can be found, so we stop.
            break

    if not pulse_durations:
        print(f"Warning: Could not find any complete pulses in {csv_path}.")
        return 0

    # Calculate the average of the durations found
    average_duration = sum(pulse_durations) / len(pulse_durations)
    # Optional: uncomment the line below to see how many pulses were averaged for each file
    # print(f"INFO: Averaged {len(pulse_durations)} pulses for {os.path.basename(csv_path)} -> {average_duration:.2f} ms")
    return average_duration

def create_comparison_chart(results):
    """Generates and displays a grouped bar chart from the results dictionary."""
    
    labels = list(results.keys())
    default_o3_vals = [res['Default O3'] for res in results.values()]
    improv_o3_vals = [res['Improved O3'] for res in results.values()]
    default_os_vals = [res['Default Os'] for res in results.values()]
    improv_os_vals = [res['Improved Os'] for res in results.values()]

    x = np.arange(len(labels))  # the label locations
    width = 0.2  # the width of the bars

    fig, ax = plt.subplots(figsize=(15, 8))

    # Create the bars for each category
    rects1 = ax.bar(x - width*1.5, default_o3_vals, width, label='Old O3', color='cornflowerblue')
    rects2 = ax.bar(x - width/2, improv_o3_vals, width, label='New O3', color='royalblue')
    rects3 = ax.bar(x + width/2, default_os_vals, width, label='Old Os', color='lightcoral')
    rects4 = ax.bar(x + width*1.5, improv_os_vals, width, label='New Os', color='firebrick')

    for i in range(len(labels)):
        labels[i] = labels[i].replace('Saleae Out/', '')

    # Add some text for labels, title and axes ticks
    ax.set_ylabel('Render Time Average [ms]', fontsize=14)
    ax.set_title('DMA2D Old Vs. New', fontsize=16, pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.legend(fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    # Add labels on top of the bars
    ax.bar_label(rects1, padding=3, fmt='%.2f', rotation=90, size=8)
    ax.bar_label(rects2, padding=3, fmt='%.2f', rotation=90, size=8)
    ax.bar_label(rects3, padding=3, fmt='%.2f', rotation=90, size=8)
    ax.bar_label(rects4, padding=3, fmt='%.2f', rotation=90, size=8)

    # Adjust layout to make room for labels and title
    fig.tight_layout()
    plt.ylim(top=ax.get_ylim()[1] * 1.15) # Add extra space at the top for labels
    plt.show()

In [ ]:
results = {}

base_names = [f.replace("_dma2d_gcc_o3.csv", "").replace("_", " ").title() for f in meas_default_o3]

for name in base_names:
    results[name] = {}

all_measurements = {
    "Default O3": meas_default_o3,
    "Improved O3": meas_improv_o3,
    "Default Os": meas_default_os,
    "Improved Os": meas_improv_os,
}

for category, file_list in all_measurements.items():
    for i, file_path in enumerate(file_list):
        test_name = base_names[i]
        duration = calculate_first_pulse_duration(file_path)
        results[test_name][category] = duration

create_comparison_chart(results)